In [2]:
import yaml

yaml_path = "ollama_config.yaml"
with open(yaml_path, "r") as file:
    yaml_config = yaml.safe_load(file)

print("Loaded YAML Data:")
yaml_config

Loaded YAML Data:


{'vectordb': [{'name': 'chroma_mpnet',
   'db_type': 'chroma',
   'client_type': 'persistent',
   'collection_name': 'huggingface_all_mpnet_base_v2',
   'embedding_model': 'huggingface_all_mpnet_base_v2',
   'path': '/home/lyb/RAG/experiments/chroma_mpnet'}],
 'node_lines': [{'node_line_name': 'retrieve_node_line',
   'nodes': [{'node_type': 'retrieval',
     'strategy': {'metrics': ['retrieval_f1',
       'retrieval_recall',
       'retrieval_precision']},
     'top_k': 3,
     'modules': [{'module_type': 'bm25'},
      {'module_type': 'vectordb', 'vectordb': 'chroma_mpnet'},
      {'module_type': 'hybrid_rrf',
       'target_modules': "('bm25', 'vectordb')",
       'rrf_k': [3, 5, 10, 100, 1000, 10000]},
      {'module_type': 'hybrid_cc',
       'target_modules': "('bm25', 'vectordb')",
       'weights': '(0.5, 0.5)'}]}]},
  {'node_line_name': 'post_retrieve_node_line',
   'nodes': [{'node_type': 'prompt_maker',
     'strategy': {'metrics': ['meteor', 'rouge', 'bert_score']},
     'm

In [2]:
test_modules = [{'module_type': 'hybrid_cc',
                    'target_modules': "('bm25', 'vectordb')",
                    'target_module_params': [{'top_k': 3}, {'top_k': 3, 'vectordb': 'chroma_mpnet'}],
                    'weights': ['(0.5, 0.5)']},
                {'module_type': 'hybrid_cc',
                    'target_modules': "('bm25', 'vectordb')",
                    'target_module_params': [{'top_k': 3}, {'top_k': 3, 'vectordb': 'chroma_mpnet'}],
                    'weights': ['(0.7, 0.3)']},
                {'module_type': 'bm25'},
                {'module_type': 'vectordb', 'vectordb': 'chroma_mpnet'},
                {'module_type': 'hybrid_rrf',
                    'target_modules': "('bm25', 'vectordb')",
                    'target_module_params': [{'top_k': 3}, {'top_k': 3, 'vectordb': 'chroma_mpnet'}],
                    'rrf_k': [5]},
                {'module_type': 'hybrid_rrf',
                    'target_modules': "('bm25', 'vectordb')",
                    'target_module_params': [{'top_k': 3}, {'top_k': 3, 'vectordb': 'chroma_mpnet'}],
                    'rrf_k': [3]},
                {'module_type': 'hybrid_rrf',
                    'target_modules': "('bm25', 'vectordb')",
                    'target_module_params': [{'top_k': 3}, {'top_k': 3, 'vectordb': 'chroma_mpnet'}],
                    'rrf_k': [10]},]

for i, module in enumerate(test_modules):
    for node in yaml_config['node_lines']:
        if node['node_line_name'] == 'retrieve_node_line':
            for node_line in node['nodes']:
                node_line['modules'] = [module]

            # Save the modified data back to a YAML file
            with open(f"./candidate_config/ollama_config_{i}.yaml", "w") as file:
                yaml.safe_dump(yaml_config, file, default_flow_style=False, sort_keys=False)

### ConfigSapce Experiments

In [2]:
from dataclasses import dataclass
from ConfigSpace import ConfigurationSpace, Constant

cs = ConfigurationSpace(yaml_config)

In [4]:
from autorag.evaluator import Evaluator
from autorag.node_line import run_node_line

node_lines = Evaluator._load_node_lines(yaml_path)

[01/22/25 09:31:48] INFO     [config.py:58] >> PyTorch version 2.5.1 available.                        ]8;id=878504;file:///home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/datasets/config.py\config.py]8;;\:]8;id=187823;file:///home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/datasets/config.py#58\58]8;;\

/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_id" in DeployedModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceLLM has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_kwargs" in HuggingFaceLLM has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fiel

In [5]:
for k,v in node_lines.items():
    print(k, v)
    print("\n")

retrieve_node_line [Node(node_type='retrieval', strategy={'metrics': ['retrieval_f1', 'retrieval_recall', 'retrieval_precision']}, node_params={'top_k': 3}, modules=[Module(module_type='bm25', module_param={}, module=<class 'autorag.nodes.retrieval.bm25.BM25'>), Module(module_type='vectordb', module_param={'vectordb': 'chroma_mpnet'}, module=<class 'autorag.nodes.retrieval.vectordb.VectorDB'>), Module(module_type='hybrid_rrf', module_param={'target_modules': ('bm25', 'vectordb'), 'rrf_k': [3, 5, 10, 100, 1000, 10000]}, module=<class 'autorag.nodes.retrieval.hybrid_rrf.HybridRRF'>), Module(module_type='hybrid_cc', module_param={'target_modules': ('bm25', 'vectordb'), 'weights': (0.5, 0.5)}, module=<class 'autorag.nodes.retrieval.hybrid_cc.HybridCC'>)], run_node=<function run_retrieval_node at 0x7f3cf5d2dea0>)]


post_retrieve_node_line [Node(node_type='prompt_maker', strategy={'metrics': ['meteor', 'rouge', 'bert_score']}, node_params={}, modules=[Module(module_type='fstring', module_pa

In [11]:
import pandas as pd

qa_data_path = "/home/lyb/RAG/data/2WikiMultihopQA/qa.parquet"
# qa_data_path = "/home/lyb/RAG/data/eli5_data/20sample/qa.parquet"


qa_data = pd.read_parquet(qa_data_path, engine="pyarrow")

In [12]:
qa_data.columns

Index(['qid', 'query', 'retrieval_gt', 'generation_gt'], dtype='object')

In [13]:
qa_data.head()

,qid,query,retrieval_gt,generation_gt
0,2WikiMultihopQA_Q0,Who is the mother of the director of film Poli...,"[[2WikiMultihopQA_C0_0, 2WikiMultihopQA_C0_1]]",Małgorzata Braunek
1,2WikiMultihopQA_Q1,"Which film came out first, Blind Shaft or The ...","[[2WikiMultihopQA_C1_0, 2WikiMultihopQA_C1_1, ...",The Mask Of Fu Manchu
2,2WikiMultihopQA_Q2,"When did John V, Prince Of Anhalt-Zerbst's fat...","[[2WikiMultihopQA_C2_0, 2WikiMultihopQA_C2_1]]",12 June 1516
3,2WikiMultihopQA_Q3,What is the award that the director of film We...,"[[2WikiMultihopQA_C3_0, 2WikiMultihopQA_C3_1]]",Myanmar Motion Picture Academy Awards
4,2WikiMultihopQA_Q4,Where was the director of film Ronnie Rocket b...,"[[2WikiMultihopQA_C4_0, 2WikiMultihopQA_C4_1]]","Missoula, Montana"


In [14]:
duplicated_columns = qa_data.columns.intersection(
                qa_data.columns
            )

In [17]:
# remove "id" and "qid" columns in duplicated_columns if they exist
duplicated_columns = duplicated_columns.drop(["id", "qid"], errors="ignore")

In [ ]:

duplicated_columns

Index(['query', 'retrieval_gt', 'generation_gt'], dtype='object')